# Paper 4 fixed-candidate development restart

Start with one real image, then 50 TRAIN-derived development questions. This notebook does **not** fit calibration policies or evaluate held-out benchmarks. It uses adopted, pinned Qwen generator and critic weights. A chat interface or successful execution does not establish scientific novelty.

Choose **Runtime → Change runtime type → GPU**. Run cells in order. A failed cell is a stop condition. There is no paid-resource purchase or automatic full-benchmark run here.

The local restart prepared 50 question records but decoded no COCO images and generated no model predictions. This notebook must establish those first real execution records. By default, data and outputs use temporary runtime storage: download the archive after every gate. Optional Drive mounting requires you to enable it and grant access. Model downloads use runtime disk.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, hashlib
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing. No predictions generated.'
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda})
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/JB_Paper4/restart_20261009')
else:
    RUN_ROOT = Path('/content/JB_Paper4/restart_20261009')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PILOT = RUN_ROOT / 'pilot50'
REPO = Path('/content/paper4_fixed_candidate_restart')
def run(args, **kwargs):
    from datetime import datetime, timezone
    command = [str(x) for x in args]
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    result = subprocess.run(command, capture_output=True, text=True, **kwargs)
    (RUN_ROOT / ('command_' + attempt + '.json')).write_text(json.dumps({'argv': command, 'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}, indent=2))
    print(result.stdout[-12000:])
    print(result.stderr[-12000:])
    result.check_returncode()
    return result
def archive_gate(label):
    import zipfile
    from datetime import datetime, timezone
    from google.colab import files
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    target = Path('/content') / ('paper4_restart_' + label + '_' + stamp + '.zip')
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RUN_ROOT.rglob('*')):
            if path.is_file() and not path.name.endswith('.part'):
                bundle.write(path, str(path.relative_to(RUN_ROOT)))
    print('Download required for retention:', target, target.stat().st_size)
    files.download(str(target))


## Retrieve the exact corrected source

This notebook checks out a frozen code commit. A pre-existing checkout with a different commit or local tracked-file changes is rejected. Keep separate output directories for the one-image and 50-question runs.


In [ ]:
SOURCE_COMMIT = 'e2a758fe6918a82a4d7a7776eb90f3b5eed0694d'
REPOSITORY = 'https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
if not REPO.exists():
    run(['git', 'clone', REPOSITORY, REPO])
    run(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT, 'Existing checkout differs; use a fresh runtime or inspect it before replacing.'
tracked = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True)
assert not tracked.strip(), 'Tracked source changed; do not silently run modified code.'
run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[vlm,dev]', 'transformers==4.57.1'])
import importlib.metadata
print({name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'accelerate', 'bitsandbytes']})
(RUN_ROOT / 'source_commit.txt').write_text(SOURCE_COMMIT + '\n')


## Prepare the frozen development data

The official A-OKVQA archive is selectively read for TRAIN annotations. There are 17,056 training questions referring to 16,540 image IDs. Only 50 selected COCO TRAIN JPEGs are downloaded, with original geometry and hashes retained. Reference answers stay in a separate offline file. The explicitly selected S3 path-style HTTPS route addresses the same official `images.cocodataset.org` bucket; it avoids the observed custom-host certificate mismatch while preserving normal certificate checks. Its selected URLs and verified download provenance are recorded.

This step stops after three consecutive image-download failures. Retry the same command to resume when connectivity is restored; frozen identities and verified image hashes cannot change. The inference gate validates the complete selected 50-image inventory even when only one question is requested.


In [ ]:
run([sys.executable, REPO / 'scripts/prepare_restart_pilot.py', '--out-dir', PILOT,
     '--download-annotations', '--download-images', '--image-transport', 'coco-s3-path', '--timeout', '30'], cwd=REPO)
run([sys.executable, REPO / 'scripts/restart_preflight.py', '--pilot-dir', PILOT,
     '--out', RUN_ROOT / 'preflight.json'], cwd=REPO)
preflight = json.loads((RUN_ROOT / 'preflight.json').read_text())
assert preflight['status'] == 'READY_FOR_ONE_IMAGE_GATE', preflight
print('Data and runtime preflight passed. This is not a model result.')


Download the data/preflight archive even if the preceding cell failed. The archive retains observed command errors and prepared files; it does not turn a failed gate into a successful experiment. Temporary runtime storage can disappear after disconnect.

In [ ]:
archive_gate('data_preflight')


## One-image execution gate

The generator loads Qwen2.5-VL-3B-Instruct and the critic loads Qwen2.5-1.5B-Instruct at the locked revisions. Requested NF4 must be confirmed. Each selected question produces one answer and ordered context; the likelihood and critic variants reuse that exact candidate.

Raw generator text, parsing/grounding diagnostics, repeated and unknown citation IDs, evidence, image processing geometry and critic continuation scores are retained. Measured resource sessions are separate from immutable resume identity. Live evidence capture is recorded; it is not an immutable upstream knowledge release.

On out-of-memory or provider failure, inspect archived errors. Do not silently lower resolution, remove a source or replace a model while retaining the same run identity. Change the declared experiment and output path if a revised configuration is approved.


In [ ]:
run([sys.executable, REPO / 'scripts/run_fixed_candidate_pilot.py', '--pilot-dir', PILOT,
     '--out-dir', RUN_ROOT / 'one_image', '--max-samples', '1'], cwd=REPO)
print('One-image command completed. Inspect its raw records before the 50-question cell.')


In [ ]:
archive_gate('one_image')


## Bounded 50-question development run

Run after the one-image gate succeeds. These questions test execution integrity and development behavior; they are not a held-out test set. Exclude all 50 selected image groups from future calibration pools.


In [ ]:
run([sys.executable, REPO / 'scripts/run_fixed_candidate_pilot.py', '--pilot-dir', PILOT,
     '--out-dir', RUN_ROOT / 'pilot50', '--max-samples', '50'], cwd=REPO)
print('Bounded command completed. No calibration or benchmark score was produced.')


In [ ]:
archive_gate('pilot50')


## Inventory the retained artifacts

Review completion counts and raw archived records. A command exit alone is not evidence of accuracy or critic superiority. Resource sessions distinguish success, invalid candidates and runtime failures.

After this gate: freeze the protocol, prepare image-disjoint calibration-fit/threshold subsets from TRAIN, fit likelihood and critic policies separately, lock them, then evaluate held-out A-OKVQA, paired uncertainty, component controls and independent OK-VQA replication. Blinded human evidence annotation is still required for grounding claims. The current runner deliberately does not perform these steps.


In [ ]:
inventory = []
for path in sorted(RUN_ROOT.rglob('*')):
    if path.is_file() and not path.name.endswith('.part'):
        inventory.append({'path': str(path.relative_to(RUN_ROOT)), 'bytes': path.stat().st_size})
(RUN_ROOT / 'retained_artifact_inventory.json').write_text(json.dumps(inventory, indent=2) + '\n')
print(json.dumps([r for r in inventory if not r['path'].endswith('.jpg')], indent=2))
print('Outputs retained at:', RUN_ROOT)
